# Task 05 — Deep Learning / NLP Text Classifier

**Project:** Movie Review Sentiment Classification using TensorFlow/Keras

## Objective
Build a deep-learning NLP model that classifies a movie review as **Positive** or **Negative** and reports the model's confidence.

## What this notebook covers
1. Load and inspect a real text dataset.
2. Prepare and pad tokenized review sequences.
3. Build a multi-layer neural network with an Embedding layer, Batch Normalization, and Dropout.
4. Train with validation monitoring and Early Stopping.
5. Plot training/validation accuracy and loss.
6. Evaluate the model on unseen test data.
7. Predict the sentiment and confidence of custom text.

**Dataset:** IMDB movie reviews provided by `tensorflow.keras.datasets`. Labels are binary: `0 = Negative`, `1 = Positive`.

> Run the cells from top to bottom. The first run downloads the IMDB dataset and may take a few minutes depending on your internet connection and hardware.

## 1. Install and import dependencies

If the libraries are already installed in your environment, the installation cell can be skipped.

In [ ]:
# Uncomment this line in a fresh Google Colab/Jupyter environment if needed:
# %pip install -q tensorflow scikit-learn matplotlib seaborn

import re
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import layers, models, callbacks
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("Available devices:", tf.config.list_physical_devices())

## 2. Load the IMDB dataset

The dataset contains 25,000 training reviews and 25,000 test reviews. Each review is represented as a sequence of integer word IDs. We keep the 10,000 most frequent words.

In [ ]:
VOCAB_SIZE = 10_000
MAX_LENGTH = 250

(train_sequences, y_train), (test_sequences, y_test) = imdb.load_data(
    num_words=VOCAB_SIZE,
    seed=SEED
)

print("Training reviews:", len(train_sequences))
print("Test reviews:", len(test_sequences))
print("Example encoded review:", train_sequences[0][:20])
print("Example label:", y_train[0], "(1 = Positive, 0 = Negative)")

## 3. Prepare the input sequences

Reviews have different lengths. Padding makes each review the same length so it can be processed in batches by the neural network. Reviews longer than `MAX_LENGTH` are truncated.

In [ ]:
X_train = pad_sequences(
    train_sequences, maxlen=MAX_LENGTH, padding="post", truncating="post"
)
X_test = pad_sequences(
    test_sequences, maxlen=MAX_LENGTH, padding="post", truncating="post"
)

y_train = np.asarray(y_train).astype("float32")
y_test = np.asarray(y_test).astype("float32")

print("Training input shape:", X_train.shape)
print("Test input shape:", X_test.shape)

## 4. Build the deep-learning model

The model uses:
- **Embedding:** learns a dense vector representation for each word ID.
- **Global Average Pooling:** combines word representations into one review vector.
- **Dense layers:** learn patterns useful for sentiment classification.
- **Batch Normalization and Dropout:** help stabilize training and reduce overfitting.
- **Sigmoid output:** returns a probability between 0 and 1.

In [ ]:
model = models.Sequential([
    layers.Input(shape=(MAX_LENGTH,)),
    layers.Embedding(input_dim=VOCAB_SIZE, output_dim=64, mask_zero=True),
    layers.GlobalAveragePooling1D(),

    layers.Dense(128, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.40),

    layers.Dense(64, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.30),

    layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

## 5. Train the model

Early Stopping monitors validation loss. If validation loss does not improve for three epochs, training stops and the best model weights are restored.

In [ ]:
early_stopping = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

history = model.fit(
    X_train,
    y_train,
    epochs=12,
    batch_size=128,
    validation_split=0.20,
    callbacks=[early_stopping],
    verbose=1
)

## 6. Plot training and validation curves

These plots show how accuracy and loss change during training. A large gap between training and validation performance can indicate overfitting.

In [ ]:
history_dict = history.history
epochs_ran = range(1, len(history_dict["loss"]) + 1)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs_ran, history_dict["accuracy"], marker="o", label="Training accuracy")
plt.plot(epochs_ran, history_dict["val_accuracy"], marker="o", label="Validation accuracy")
plt.title("Training and Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(epochs_ran, history_dict["loss"], marker="o", label="Training loss")
plt.plot(epochs_ran, history_dict["val_loss"], marker="o", label="Validation loss")
plt.title("Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Evaluate on unseen test reviews

The test set was kept separate from training and validation. The classification report provides precision, recall, and F1-score for both sentiment classes.

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")

test_probabilities = model.predict(X_test, batch_size=256, verbose=0).ravel()
test_predictions = (test_probabilities >= 0.5).astype("int32")

print("\nClassification report:")
print(classification_report(
    y_test.astype("int32"),
    test_predictions,
    target_names=["Negative", "Positive"],
    digits=4
))

cm = confusion_matrix(y_test.astype("int32"), test_predictions)
display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Negative", "Positive"]
)
display.plot(cmap="Blues", values_format="d")
plt.title("Confusion Matrix — Test Set")
plt.show()

## 8. Predict sentiment for new text

The Keras IMDB dataset provides a word-to-index dictionary. The helper function below converts ordinary text into the same integer format used by the training data, pads it, and returns a predicted label with a confidence score.

The confidence is the model's estimated probability for its predicted class; it is not a guarantee that the prediction is correct.

In [ ]:
word_index = imdb.get_word_index()

def encode_review(text, max_length=MAX_LENGTH):
    # Keep simple alphabetic tokens and convert to lowercase.
    tokens = re.findall(r"[a-z']+", text.lower())

    # IMDB reserves 0 for padding, 1 for start, and 2 for unknown words.
    encoded = [1]
    for token in tokens:
        index = word_index.get(token)
        encoded.append(index + 3 if index is not None and index + 3 < VOCAB_SIZE else 2)

    return pad_sequences(
        [encoded], maxlen=max_length, padding="post", truncating="post"
    )

def predict_sentiment(review):
    if not isinstance(review, str) or not review.strip():
        raise ValueError("Please enter a non-empty review.")

    sequence = encode_review(review)
    positive_probability = float(model.predict(sequence, verbose=0)[0][0])

    if positive_probability >= 0.5:
        label = "Positive"
        confidence = positive_probability
    else:
        label = "Negative"
        confidence = 1.0 - positive_probability

    print("Review:", review)
    print("Predicted sentiment:", label)
    print(f"Confidence: {confidence * 100:.2f}%")
    print(f"Positive probability: {positive_probability * 100:.2f}%")
    return label, confidence

## 9. Try sample reviews

Run these examples, then replace the text with your own review.

In [ ]:
sample_reviews = [
    "This movie was wonderful. The acting was brilliant and I enjoyed every minute.",
    "The story was boring and the performances were disappointing.",
    "A memorable film with great characters and an excellent ending."
]

for review in sample_reviews:
    print("-" * 70)
    predict_sentiment(review)

## 10. Save the trained model

This saves the trained neural network in Keras format. The file is created in the notebook's current working directory.

In [ ]:
model.save("imdb_sentiment_classifier.keras")
print("Model saved as imdb_sentiment_classifier.keras")

## 11. Conclusion

In this project, a deep-learning NLP pipeline was developed to classify IMDB movie reviews as positive or negative. The notebook prepares tokenized text sequences, trains an embedding-based neural network with Batch Normalization, Dropout, and Early Stopping, visualizes training/validation curves, evaluates test performance, and demonstrates predictions with confidence scores.

### Expected submission proof
Submit this completed `.ipynb` notebook with:
- Model summary
- Training output
- Accuracy and loss curves
- Test-set evaluation and confusion matrix
- Sample inference outputs with confidence scores

**Important:** Run all cells before submission so the notebook contains your actual training graphs, evaluation metrics, and prediction outputs. Do not report a test accuracy until you have run the notebook in your environment.